# Análisis Exploratorio de Datos (EDA) Clínicos - Cohorte METABRIC

**Trabajo de Fin de Máster:** Supervivencia en Cáncer de Mama y Pulmón
**Autor:** Julio Úbeda Quesada

Este *notebook* contiene el Análisis Exploratorio de Datos (EDA) para el conjunto de datos clínicos de la cohorte **METABRIC** (cáncer de mama). El objetivo de esta fase es comprender la estructura de las variables, evaluar la calidad de los datos (valores nulos, distribuciones) y explorar las variables objetivo fundamentales para el análisis de supervivencia: el tiempo de seguimiento (`OS_MONTHS`) y el estado vital/evento (`OS_STATUS`).

In [ ]:
# Importación de librerías necesarias
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

## 1. Estrategia de Adquisición de Datos

Para evitar lidiar con la complejidad de la API de GDC o las dependencias de R (`TCGAbiolinks`), extraeremos los datos curados directamente desde el *Datahub* público de [cBioPortal Datasets](https://www.cbioportal.org/datasets). Esto garantiza que las variables clínicas de supervivencia (tiempo al evento, estado vital) y los perfiles genómicos vengan en un formato tabular (`.txt`) listo para usar en `pandas`.

Los identificadores de los estudios son:

* **METABRIC (Cáncer de mama):** `brca_metabric`: 
  * C:\Users\usuario\Documents\GitHub\TFM---Analisis-de-supervivencia-del-cancer\Modulo 3 - Diseño e implementacion del trabajo\data\brca_metabric_clinical_data.tsv
* **TCGA-BRCA (Cáncer de mama):** `brca_tcga_pan_can_atlas_2018` 
  * C:\Users\usuario\Documents\GitHub\TFM---Analisis-de-supervivencia-del-cancer\Modulo 3 - Diseño e implementacion del trabajo\data\brca_tcga_gdc_clinical_data.tsv
* **TCGA-LUAD (Adenocarcinoma de pulmón):** `luad_tcga_pan_can_atlas_2018`
* **TCGA-LUSC (Carcinoma escamoso de pulmón):** `lusc_tcga_pan_can_atlas_2018`
  * C:\Users\usuario\Documents\GitHub\TFM---Analisis-de-supervivencia-del-cancer\Modulo 3 - Diseño e implementacion del trabajo\data\nsclc_ctdx_msk_2022_clinical_data.tsv

## 2. Pipeline en Python para Extracción y Carga Local

Con los archivos ya descargados en tu disco duro, este nuevo pipeline se encargará únicamente de descomprimirlos (si no lo has hecho ya) y de cargar de forma segura los archivos clínicos y de expresión en DataFrames de `pandas`.

Hemos eliminado toda la lógica de conexión web para garantizar que el código se ejecute sin problemas.

In [2]:
import os
import tarfile
import pandas as pd

def extraer_y_cargar_estudio(study_id, data_dir="data"):
    """
    Descomprime el archivo .tar.gz local y carga los datos de supervivencia y expresión.
    """
    tar_path = os.path.join(data_dir, f"{study_id}.tar.gz")
    extract_path = os.path.join(data_dir, study_id)

    # 1. Extraer si el archivo .tar.gz existe y aún no se ha descomprimido
    if not os.path.exists(extract_path):
        if not os.path.exists(tar_path):
            raise FileNotFoundError(f"No se encontró el archivo: {tar_path}. ¡Asegúrate de descargarlo manualmente en la carpeta '{data_dir}'!")
            
        print(f"Descomprimiendo {study_id}...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=extract_path)
        print(f"Descompresión completada en {extract_path}")
    else:
        print(f"El estudio {study_id} ya estaba descomprimido.")

    # 2. Cargar datos clínicos (Supervivencia)
    clin_path = os.path.join(extract_path, "data_clinical_patient.txt")
    if not os.path.exists(clin_path):
        # A veces el archivo clínico puede tener un prefijo ligeramente distinto
        clin_file = [f for f in os.listdir(extract_path) if "clinical_patient" in f][0]
        clin_path = os.path.join(extract_path, clin_file)
        
    df_clin = pd.read_csv(clin_path, sep='\t', skiprows=4) 
    
    # 3. Cargar datos de expresión génica
    try:
        # Buscamos dinámicamente el archivo de mRNA
        expr_file = [f for f in os.listdir(extract_path) if "data_mrna" in f and f.endswith(".txt")][0]
        expr_path = os.path.join(extract_path, expr_file)
        print(f"Cargando expresión génica desde: {expr_file} (Esto puede tardar...)")
        df_expr = pd.read_csv(expr_path, sep='\t')
    except IndexError:
        print(f"Advertencia: No se encontró archivo de expresión mRNA (.txt) para {study_id}.")
        df_expr = pd.DataFrame() # Retorna DataFrame vacío para evitar cuelgues
    
    return df_clin, df_expr

# --- EJECUCIÓN DEL PIPELINE LOCAL ---

# Estos nombres deben coincidir con los archivos .tar.gz que descargaste (sin la extensión)
estudios = [
    "brca_metabric", 
    "brca_tcga_pan_can_atlas_2018", 
    "luad_tcga_pan_can_atlas_2018"
]

datos_proyecto = {}
directorio_base = "data"

for estudio in estudios:
    try:
        df_clin, df_expr = extraer_y_cargar_estudio(estudio, data_dir=directorio_base)
        datos_proyecto[estudio] = {'clinico': df_clin, 'expresion': df_expr}
        
        print(f"--- RESULTADOS PARA {estudio.upper()} ---")
        print(f"Pacientes clínicos: {df_clin.shape[0]}")
        print(f"Variables/Sondas de expresión: {df_expr.shape[0] if not df_expr.empty else 0}")
        print("-" * 40)
    except Exception as e:
        print(f"Error procesando {estudio}: {str(e)}\n")

Error procesando brca_metabric: No se encontró el archivo: data\brca_metabric.tar.gz. ¡Asegúrate de descargarlo manualmente en la carpeta 'data'!

Error procesando brca_tcga_pan_can_atlas_2018: No se encontró el archivo: data\brca_tcga_pan_can_atlas_2018.tar.gz. ¡Asegúrate de descargarlo manualmente en la carpeta 'data'!

Error procesando luad_tcga_pan_can_atlas_2018: No se encontró el archivo: data\luad_tcga_pan_can_atlas_2018.tar.gz. ¡Asegúrate de descargarlo manualmente en la carpeta 'data'!




## 4. Siguientes pasos: Preprocesamiento e Integración

Una vez que tengas los `DataFrames` listos, debes continuar con las actividades de tu Fase 2. Según tu planificación metodológica, deberás aplicar[cite: 211]:

* **Limpieza Clínica:** Filtrar pacientes que no tengan datos de tiempo de seguimiento (`OS_MONTHS` o similar) y estado del evento (censura/muerte).
* **Transposición y Fusión:** Los archivos de expresión génica suelen tener los genes en filas y los pacientes en columnas. Deberás transponer el DataFrame para que los pacientes sean las filas y luego hacer un `merge` (unión) con los datos clínicos usando el ID del paciente.
* [cite_start]**Filtrado por Varianza:** Eliminar los genes con baja varianza, ya que no aportan información predictiva para tus modelos[cite: 211].
* [cite_start]**Armonización (Batch Effect):** METABRIC utiliza microarrays de expresión génica y TCGA utiliza RNA-seq[cite: 444, 557]. No puedes comparar los valores brutos directamente. [cite_start]Será necesario escalar y estandarizar (ej. `StandardScaler` de scikit-learn) los datos genómicos dentro de cada cohorte de forma independiente antes de intentar la validación cruzada externa de tus modelos[cite: 557, 564].